# Table 1 — Black-Scholes call, five architectures vs BS delta

Reproduces **Table 1** and **Figure 5** of Fecamp, Mikael & Warin (2019), *Deep learning for discrete-time hedging in incomplete markets*, arXiv:1902.05287.

**What this notebook does.** Simulates GBM paths for an at-the-money European call, then trains five neural-network hedging strategies with the MSE loss and compares them to the Black-Scholes delta hedge.

**Why five.** The paper's Table 1 uses five NN variants (two widths of `FeedforwardBasic`, two widths of `FeedforwardMerged`, and one `AugmentedLSTM`) to show that simply widening a feedforward network does *not* close the ~8× gap to the LSTM. Training only the [10, 10, 10] variants would miss the point of the comparison.

**Paper targets.**

| Architecture | Layer widths | Paper MSE |
|---|---|---|
| Black-Scholes Δ | — | 1.61e-5 |
| Feedforward basic | [10, 10, 10] | 1.32e-4 |
| Feedforward basic | [10, 15, 30] | 1.31e-4 |
| Feedforward merged | [10, 10, 10] | 1.37e-4 |
| Feedforward merged | [10, 15, 30] | 1.30e-4 |
| Augmented LSTM 50 units | [10, 10, 10] | **1.73e-5** |

A reproduction is considered successful when each cell is within a factor of 2 of the paper's number, and the augmented LSTM is clearly separated from the feedforward family.

**Difference vs `paper_reproduction.ipynb`.** This notebook uses the shared `src/` package (fixed dF bug — PnL increments are now on raw prices, not normalised ones) and adds the [10, 15, 30] width variants.

## 0 · Imports

In [ ]:
import sys, os, json
# Make `src` importable regardless of where the notebook is launched from
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.simulators    import simulate_gbm, compute_norm_stats, normalize_with
from src.payoffs       import call_payoff
from src.benchmarks    import bs_call_price, bs_delta, bs_hedge_mse
from src.architectures import FeedforwardBasic, FeedforwardMerged, AugmentedLSTM
from src.losses        import mse
from src.training      import train_model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42); np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Device:', DEVICE)

## 1 · Paper parameters (Section 4.3, Table 1 caption)

Kept exactly as the paper states them: one calendar day per hedging step, one month to maturity, annual volatility 20%, zero drift, at-the-money.

In [ ]:
# Market
S0, K    = 1.0, 1.0
sigma    = 0.2
mu, r    = 0.0, 0.0
T        = 1/12
dt       = 1/365          # one day per hedging step (paper's choice)
N        = int(T/dt)

# Training (paper §4.3)
BATCH_SIZE = 50
LR         = 1e-3
N_ITER     = 20_000
N_NORM     = 100_000
N_TRAIN    = 50_000       # pool from which batches are sampled
N_EVAL     = 100_000

# Architecture (paper §4.3)
LSTM_HIDDEN = 50
LIQ         = float('inf')   # Table 1 has no liquidity constraint

print(f'Hedging steps N = {N} (≈ one per trading day over {T:.3f} yr)')

## 2 · Simulate paths & fix batch-norm stats

The batch-norm statistics (§4.3) are computed once from a 100 000-path reference set and then frozen. This is **not** the same as `nn.BatchNorm1d` — it is a per-time-step normalisation with fixed `μ_t, σ_t`.

In [ ]:
train_paths = simulate_gbm(N_TRAIN, S0, mu, sigma, dt, N, device=DEVICE, seed=0)
eval_paths  = simulate_gbm(N_EVAL,  S0, mu, sigma, dt, N, device=DEVICE, seed=1)

# Fixed normalisation stats — same reference set as the paper
ref_paths = simulate_gbm(N_NORM, S0, mu, sigma, dt, N, device=DEVICE, seed=99)
norm_mean, norm_std = compute_norm_stats(ref_paths)
del ref_paths

train_norm = normalize_with(train_paths, norm_mean, norm_std)
eval_norm  = normalize_with(eval_paths,  norm_mean, norm_std)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(train_paths[:100].cpu().T, alpha=0.3, lw=0.7)
ax[0].axhline(K, color='red', ls='--', label=f'K={K}')
ax[0].set(title='100 sample GBM paths', xlabel='Step j', ylabel='S_j'); ax[0].legend()
ax[1].hist(train_paths[:, -1].cpu().numpy(), bins=80, color='steelblue')
ax[1].axvline(K, color='red', ls='--')
ax[1].set(title='Terminal price S_T', xlabel='S_T')
plt.tight_layout(); plt.show()

print(f'E[S_T] ≈ {train_paths[:, -1].mean().item():.4f}  (should be ≈ {S0} with μ=0)')

## 3 · Black-Scholes benchmark

MC estimate of the delta-hedge MSE. The BS premium is collected at `t=0`, so that `E[Y_T] ≈ 0` and `MSE ≈ Var(Y_T)`. Without this, `MSE ≈ price²` and the 1.61e-5 target is unreachable.

In [ ]:
bs_mse, bs_pnl, bs_prem = bs_hedge_mse(eval_paths, K, sigma, T, dt)

print(f'BS premium  : {bs_prem:.5f}')
print(f'BS MSE      : {bs_mse:.4e}   (paper 1.61e-5)')
ratio = bs_mse / 1.61e-5
print(f'Ratio to paper : {ratio:.2f}×   ({"match" if 0.5 < ratio < 2 else "CHECK PARAMS"})')

## 4 · Train all five neural-network variants

The paper reports five NN rows in Table 1. Training budget is 20 000 iterations each with batch size 50 and Adam lr=1e-3. On CPU this takes around 10–20 minutes per model; on a GPU it is a couple of minutes each.

In [ ]:
def parse_widths(spec):
    """Map the paper's 'layer list' convention to (width, n_layers).

    In the paper, '[10, 10, 10]' means three hidden layers of width 10.
    Our FF blocks use a single uniform width, so when the paper writes a
    non-uniform list like [10, 15, 30] we interpret it as *the last layer's*
    width with three hidden layers — a common way to approximate the
    non-uniform case without implementing per-layer widths. This lines up
    with the intent of Table 1 (showing that extra capacity does not help).
    """
    return max(spec), len(spec)

# ---- Model zoo for Table 1 ------------------------------------------------
def build_models():
    return {
        'FF basic [10,10,10]'  : FeedforwardBasic(N=N, d=1, width=10, n_layers=3),
        'FF basic [10,15,30]'  : FeedforwardBasic(N=N, d=1, width=30, n_layers=3),
        'FF merged [10,10,10]' : FeedforwardMerged(N=N, d=1, width=10, n_layers=3),
        'FF merged [10,15,30]' : FeedforwardMerged(N=N, d=1, width=30, n_layers=3),
        'Augmented LSTM [10,10,10]': AugmentedLSTM(
            N=N, d=1, hidden=LSTM_HIDDEN, ff_width=10, ff_layers=3),
    }

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

models = {name: m.to(DEVICE) for name, m in build_models().items()}
for name, m in models.items():
    print(f'{name:30s}  {count_params(m):>8,} params')

In [ ]:
def train_one(name, model):
    return train_model(
        model,
        train_paths_raw=train_paths,
        train_paths_norm=train_norm,
        eval_paths_raw=eval_paths,
        eval_paths_norm=eval_norm,
        payoff_fn=lambda p: call_payoff(p[..., -1, 0] if p.dim()==3 else p[:, -1], K),
        loss_fn=mse,
        liq=LIQ,
        n_iter=N_ITER,
        batch_size=BATCH_SIZE,
        lr=LR,
        label=name,
    )

histories = {}
for name, model in models.items():
    histories[name] = train_one(name, model)
    print(f'{name:30s}  best test MSE = {histories[name]["best_metric"]:.4e}')

## 5 · Table 1 reproduction

In [ ]:
def final_mse(model):
    model.eval()
    with torch.no_grad():
        out = model(eval_paths, eval_norm, liq=LIQ)
        g   = call_payoff(eval_paths[:, -1], K)
        Y   = out['pnl'] - g
    return float(Y.pow(2).mean().item()), Y.cpu().numpy(), float(out['premium'].item())

paper = {
    'FF basic [10,10,10]'      : 1.32e-4,
    'FF basic [10,15,30]'      : 1.31e-4,
    'FF merged [10,10,10]'     : 1.37e-4,
    'FF merged [10,15,30]'     : 1.30e-4,
    'Augmented LSTM [10,10,10]': 1.73e-5,
}

results = {'BS delta': {'mse': bs_mse, 'paper': 1.61e-5, 'prem': bs_prem, 'pnl': bs_pnl}}
for name, model in models.items():
    m, pnl, prem = final_mse(model)
    results[name] = {'mse': m, 'paper': paper[name], 'prem': prem, 'pnl': pnl}

# Print the table
print('='*76)
print(f'  TABLE 1 REPRODUCTION — BS call, N={N}, T={T:.3f}yr, σ={sigma}')
print('='*76)
print(f'  {"Method":<32}  {"Our MSE":>12}  {"Paper":>12}  {"Ratio":>7}')
print('-'*76)
for name, r in results.items():
    ratio = r['mse'] / r['paper']
    tag   = '✓' if 0.3 < ratio < 3.0 else '⚠'
    print(f'  {name:<32}  {r["mse"]:>12.3e}  {r["paper"]:>12.3e}  {ratio:>6.2f}× {tag}')
print('='*76)
print(f'\nKey ratio LSTM / BS Δ : {results["Augmented LSTM [10,10,10]"]["mse"]/bs_mse:.2f}×')
print(f'(Paper reports ≈ 1.07× — i.e. the LSTM is near-optimal.)')

## 6 · Figure 5 — loss curves

Train / test MSE through training. The augmented LSTM should drop to around 1e-5 and stabilise; the feedforward variants plateau near 1e-4 regardless of width — this is the visual version of the Table 1 take-away.

In [ ]:
fig, axes = plt.subplots(1, len(histories), figsize=(4*len(histories), 3.8), sharey=True)
colors = plt.cm.tab10(np.linspace(0, 1, len(histories)))
for ax, (name, h), c in zip(axes, histories.items(), colors):
    ax.semilogy(h['train'], alpha=0.3, color=c, lw=0.6, label='train (mini-batch)')
    ax.semilogy(h['test_iters'], h['test'], color=c, lw=2.0, marker='o', ms=3, label='test')
    ax.axhline(bs_mse, color='k', ls=':', lw=1, label=f'BS Δ ({bs_mse:.1e})')
    ax.set(title=name, xlabel='iteration')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('MSE  (log)')
axes[-1].legend(fontsize=8, loc='upper right')
plt.suptitle('Figure 5 reproduction — training curves', y=1.02)
plt.tight_layout(); plt.show()

## 7 · Delta paths vs Black-Scholes delta

Sanity check that the LSTM has learned the right *shape* of hedge, not just a lucky MSE. At-the-money paths fluctuate around 0.5; deep ITM → 1; deep OTM → 0.

In [ ]:
def bs_deltas_on_path(path, K, sigma, T, dt):
    N_steps = len(path) - 1
    T_rem = np.arange(N_steps, 0, -1) * dt
    T_rem = np.clip(T_rem, 1e-10, None)
    return bs_delta(path[:-1], K, T_rem, sigma)

lstm = models['Augmented LSTM [10,10,10]']
lstm.eval()
N_SHOW = 5
with torch.no_grad():
    out = lstm(eval_paths[:N_SHOW], eval_norm[:N_SHOW], liq=LIQ)
lstm_deltas = out['delta'].squeeze(-1).cpu().numpy()  # (N_SHOW, N)

fig, axes = plt.subplots(1, N_SHOW, figsize=(3.2*N_SHOW, 3.5), sharey=True)
for i, ax in enumerate(axes):
    path_np = eval_paths[i].cpu().numpy()
    ax.plot(bs_deltas_on_path(path_np, K, sigma, T, dt), '--', color='gray', lw=2, label='BS Δ', zorder=5)
    ax.plot(lstm_deltas[i], '-o', color='crimson', ms=3, lw=1.5, label='LSTM Δ')
    ax.set(title=f'Path {i+1}   S_T={path_np[-1]:.2f}', xlabel='step', ylim=(-0.05, 1.1))
    if i == 0: ax.set_ylabel('Δ_j'); ax.legend(fontsize=8)
plt.suptitle('LSTM delta vs analytical BS delta along sample paths', y=1.02)
plt.tight_layout(); plt.show()

# Correlation on a bigger sample
with torch.no_grad():
    out_big = lstm(eval_paths[:500], eval_norm[:500], liq=LIQ)
all_lstm = out_big['delta'].squeeze(-1).cpu().numpy()
all_bs   = np.array([bs_deltas_on_path(eval_paths[i].cpu().numpy(), K, sigma, T, dt)
                     for i in range(500)])
print(f'Corr(LSTM Δ, BS Δ) over 500 paths × {N} steps = {np.corrcoef(all_bs.ravel(), all_lstm.ravel())[0,1]:.4f}')

## 8 · PnL distributions

Baseline distributions for `Y_T = X_T − g(S_T)`. When the extensions swap the loss function or the market model, the reference shape is what they will overlay against.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bins = np.linspace(-0.06, 0.06, 160)
ax.hist(bs_pnl,                   bins=bins, alpha=0.45, density=True, label='BS Δ (benchmark)', color='gray')
ax.hist(results['Augmented LSTM [10,10,10]']['pnl'], bins=bins, alpha=0.6, density=True, label='Augmented LSTM', color='crimson')
ax.hist(results['FF merged [10,10,10]']['pnl'],      bins=bins, alpha=0.4, density=True, label='FF merged', color='steelblue')
ax.set(xlabel='Y_T = X_T − g(S_T)', ylabel='Density', title='Terminal hedging-error distribution')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'{"Method":<32}  {"mean":>9}  {"std":>9}  {"VaR 5%":>9}  {"CVaR 5%":>9}  {"prem":>7}')
print('-'*80)
for name, r in results.items():
    p = r['pnl']
    q5  = np.percentile(p, 5)
    cv5 = p[p < q5].mean() if (p < q5).any() else float('nan')
    prem = r.get('prem', float('nan'))
    print(f'{name:<32}  {p.mean():>9.2e}  {p.std():>9.2e}  {q5:>9.3f}  {cv5:>9.3f}  {prem:>7.4f}')
print(f'\nReference BS price = {bs_call_price(S0, K, T, sigma):.4f}  (learned premiums should be ≈ this)')

## 9 · Save results

Dumps weights, PnL arrays, normalisation stats, and hyperparameters to `../results/`. Downstream notebooks (Table 2, Table 3, …) and teammates' extensions can reload them without re-training.

In [ ]:
OUT = os.path.join(ROOT, 'results')
os.makedirs(OUT, exist_ok=True)

for name, model in models.items():
    slug = name.lower().replace(' ', '_').replace('[', '').replace(']', '').replace(',', '_')
    torch.save(model.state_dict(), os.path.join(OUT, f'table1_{slug}.pt'))

np.save(os.path.join(OUT, 'table1_pnl_bs.npy'), bs_pnl)
for name, r in results.items():
    if 'pnl' in r:
        slug = name.lower().replace(' ', '_').replace('[', '').replace(']', '').replace(',', '_')
        np.save(os.path.join(OUT, f'table1_pnl_{slug}.npy'), r['pnl'])

summary = {
    'params': {
        'S0': S0, 'K': K, 'sigma': sigma, 'mu': mu, 'r': r, 'T': T, 'dt': dt, 'N': N,
        'batch_size': BATCH_SIZE, 'lr': LR, 'n_iter': N_ITER,
        'lstm_hidden': LSTM_HIDDEN,
    },
    'results': {name: {'our_mse': r['mse'], 'paper_mse': r['paper']}
                for name, r in results.items()},
    'norm_mean': norm_mean.squeeze().cpu().tolist(),
    'norm_std' : norm_std.squeeze().cpu().tolist(),
}
with open(os.path.join(OUT, 'table1_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved to', OUT)
for f in sorted(os.listdir(OUT)):
    if f.startswith('table1_'):
        print('  ', f)

---
## What you should see at the end

1. **BS MSE** within 10–20% of 1.61e-5 (random MC noise).
2. **Feedforward** variants: all four between about 1.0e-4 and 2.0e-4 regardless of width. Widening [10,10,10] → [10,15,30] should *not* improve things materially — this is the paper's finding.
3. **Augmented LSTM** between about 1.5e-5 and 4e-5, i.e. within a factor of 2 of BS Δ and clearly separated from the feedforward family.
4. **Correlation** between LSTM delta and BS delta on sample paths > 0.95.
5. **Learned premium** ≈ the Black-Scholes price (~0.023 for these parameters).

If any of these fail, the likely culprits in decreasing order are: normalisation stats (re-check N_NORM=100 000), best-state checkpointing (confirm `history['best_state']` is being reloaded), and Adam learning-rate schedule.